# 03 — Path → IL · Task 14.4

Implement and exercise the IL formulas (`spec.md` §3.1 + §3.2 + payout cap).

Three layers:

1. **Single position, IL vs price** — V_hold, V_lp, raw IL, and the FULL payout (`min(IL, MaxIL)`) across a price grid.
2. **MaxIL / V0 reference table** — regenerates the placeholder magnitudes in spec §3.2.
3. **IL distribution over Monte Carlo paths** — combine notebook 01's price simulator with notebook 02's position sampler.

All math lives in `inflexion_quant.il`; cross-check vs the Stylus contract is Phase 2 (Task 2.11).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from inflexion_quant.il import (
    compute_il, compute_max_il, compute_payout,
    entry_amounts, hold_value, lp_value, position_V0,
)
from inflexion_quant.positions import PositionMix, sample_positions
from inflexion_quant.prices import gbm_paths, kou_jump_paths, KouParams

rng = np.random.default_rng(seed=20260526)
plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': 0.3})

## A. One position — V_hold, V_lp, IL, payout across the price grid

In [ ]:
P0 = 3000.0
Pa = 2400.0           # ±20% wide for a clearly-visible curve
Pb = 3600.0
L = 1_000_000.0

a0, a1 = entry_amounts(P0, Pa, Pb, L)
V0 = position_V0(P0, Pa, Pb, L)
max_il = compute_max_il(P0, Pa, Pb, L)
print(f'V0={V0:,.0f}  amount0={a0:.3f} ETH  amount1={a1:,.0f} USDC')
print(f'MaxIL={max_il:,.0f}  ({100 * max_il / V0:.2f}% of V0)')

P_T = np.linspace(P0 * 0.4, P0 * 2.0, 600)
vh = hold_value(P_T, a0, a1)
vl = lp_value(P_T, Pa, Pb, L)
il = compute_il(P_T, Pa, Pb, L, a0, a1)
pay = compute_payout(P_T, P0, Pa, Pb, L)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(P_T, vh, label='V_hold', lw=2)
axes[0].plot(P_T, vl, label='V_lp', lw=2)
for x, lbl, ls in [(Pa, 'Pa', ':'), (P0, 'P0', '--'), (Pb, 'Pb', ':')]:
    axes[0].axvline(x, color='k', ls=ls, lw=0.6, alpha=0.5)
    axes[0].text(x, axes[0].get_ylim()[1] * 0.95, lbl, ha='center', fontsize=9)
axes[0].set(xlabel='settlement price P_T', ylabel='token1 (USDC)', title='V_hold vs V_lp')
axes[0].legend()

axes[1].plot(P_T, il, label='realised IL = max(0, V_hold − V_lp)', color='C3', lw=2)
axes[1].plot(P_T, pay, label='FULL payout = min(IL, MaxIL)', color='C2', lw=2.5, ls='--')
axes[1].axhline(max_il, color='k', ls=':', lw=0.7, alpha=0.6)
axes[1].text(P_T[10], max_il * 1.02, f'MaxIL = {max_il:,.0f}', fontsize=9)
for x, lbl in [(Pa, 'Pa'), (Pb, 'Pb')]:
    axes[1].axvline(x, color='k', ls=':', lw=0.6, alpha=0.4)
axes[1].set(xlabel='settlement price P_T', ylabel='IL  (USDC)',
            title=f'IL & payout — the cap bites past Pb (and far below Pa)')
axes[1].legend(); plt.tight_layout(); plt.show()

## B. MaxIL / V0 reference table

This regenerates (and supersedes) the placeholder table in spec.md §3.2. The numbers here are the *actual* MaxIL/V0 for centred log-symmetric ranges.

In [ ]:
rows = []
P0 = 1_000_000.0  # arbitrary — MaxIL/V0 is scale-invariant
L = 1_000_000.0
for hw in [0.02, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50, 0.80, 1.00]:
    Pa = P0 / (1 + hw)
    Pb = P0 * (1 + hw)
    V0 = position_V0(P0, Pa, Pb, L)
    mi = compute_max_il(P0, Pa, Pb, L)
    rows.append({
        'range_half_width': f'±{int(100 * hw)}%',
        'MaxIL/V0 (%)': round(100 * mi / V0, 3),
    })
tbl = pd.DataFrame(rows).set_index('range_half_width')
print(tbl)

fig, ax = plt.subplots(figsize=(8, 4.5))
hws = np.linspace(0.01, 1.2, 200)
ratios = []
for hw in hws:
    Pa = P0 / (1 + hw); Pb = P0 * (1 + hw)
    V0 = position_V0(P0, Pa, Pb, L)
    ratios.append(100 * compute_max_il(P0, Pa, Pb, L) / V0)
ax.plot(100 * hws, ratios, lw=2)
ax.set(xlabel='range half-width (%)', ylabel='MaxIL / V0  (%)',
       title='MaxIL/V0 vs range width (centred, log-symmetric)')
plt.tight_layout(); plt.show()

## C. IL distribution under simulated paths — single position

In [ ]:
# Same position as section A
P0, Pa, Pb, L = 3000.0, 2400.0, 3600.0, 1_000_000.0
V0 = position_V0(P0, Pa, Pb, L)
max_il = compute_max_il(P0, Pa, Pb, L)

# Simulate 30-day terminal prices: GBM and Kou (fat tails)
T, n_steps, n_paths = 30 / 365, 24 * 30, 20_000
sigma = 0.65
g = gbm_paths(P0, 0.0, sigma, T, n_steps, n_paths, rng)
k = kou_jump_paths(P0, 0.0, sigma, T, n_steps, n_paths,
                    KouParams(lam=80, p_up=0.4, eta_up=25, eta_down=15), rng)

pay_g = compute_payout(g[:, -1], P0, Pa, Pb, L)
pay_k = compute_payout(k[:, -1], P0, Pa, Pb, L)

print(f'{"":>15s} {"GBM":>12s} {"Kou":>12s}')
print(f'{"E[payout/V0]":>15s} {pay_g.mean()/V0:>11.2%} {pay_k.mean()/V0:>11.2%}')
print(f'{"P(payout > 0)":>15s} {(pay_g > 1e-6).mean():>11.2%} {(pay_k > 1e-6).mean():>11.2%}')
print(f'{"P(IL > MaxIL)":>15s} {(compute_il(g[:,-1], Pa, Pb, L, *entry_amounts(P0,Pa,Pb,L)) > max_il).mean():>11.2%}  {(compute_il(k[:,-1], Pa, Pb, L, *entry_amounts(P0,Pa,Pb,L)) > max_il).mean():>11.2%}')

fig, ax = plt.subplots(figsize=(11, 4.5))
bins = np.linspace(0, max_il * 1.05, 60)
ax.hist(pay_g, bins=bins, density=True, alpha=0.5, color='C0', label='GBM')
ax.hist(pay_k, bins=bins, density=True, alpha=0.5, color='C3', label='Kou (fat tails)')
ax.axvline(max_il, color='k', ls='--', lw=1, alpha=0.7, label=f'MaxIL = {max_il:,.0f}')
ax.set(xlabel='terminal payout (USDC)', ylabel='density',
       title=f'30d FULL payout distribution — V0={V0:,.0f}, ±20% range')
ax.legend(); plt.tight_layout(); plt.show()

## D. IL distribution across the position mix

Now combine: sample 2k positions from the mix, simulate one terminal price for each, compute payout/V0. This is the input shape Phase 14.5 will sum across to get the *portfolio* IL — and Phase 14.6 will stress with crash factors.

In [ ]:
P0 = 3000.0
positions = sample_positions(2_000, P0=P0, mix=PositionMix.crypto_majors(), rng=rng)

# One terminal price per position via Kou (fat tails)
T, n_steps = 30 / 365, 24 * 30
k = kou_jump_paths(P0, 0.0, 0.65, T, n_steps, len(positions),
                    KouParams(lam=80, p_up=0.4, eta_up=25, eta_down=15), rng)
P_T_each = k[:, -1]

# Compute payout per position
pay = np.array([
    float(compute_payout(P_T_each[i], P0, row.Pa, row.Pb, row.L))
    for i, row in enumerate(positions.itertuples())
])
positions['payout'] = pay
positions['payout_over_V0'] = pay / positions['V0']
positions['hit_cap'] = pay >= [compute_max_il(P0, row.Pa, row.Pb, row.L) - 1e-6
                                 for row in positions.itertuples()]

print(positions[['half_width', 'V0', 'payout', 'payout_over_V0', 'hit_cap']].describe().round(4))
print(f'\n{positions["hit_cap"].mean():.1%} of positions hit the MaxIL cap')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(positions['payout_over_V0'] * 100, bins=60, color='C2', alpha=0.8)
axes[0].set(xlabel='payout / V0  (%)', ylabel='count',
            title=f'payout/V0 distribution — 2k positions, single 30d path each')

# Scatter: half_width vs payout/V0 — wider positions can have larger IL relative to V0
axes[1].scatter(positions['half_width'] * 100, positions['payout_over_V0'] * 100,
                 alpha=0.3, s=8, c=positions['hit_cap'].map({True: 'C3', False: 'C0'}))
axes[1].set(xlabel='half-width (%)', ylabel='payout / V0  (%)', xscale='log',
             title='wider range → higher MaxIL/V0 ceiling; red = hit cap')
plt.tight_layout(); plt.show()

## E. Next

**Task 14.5 — Portfolio waterfall.** For each Monte Carlo scenario (many concurrent positions, many paths), apply the PARTIAL split: LP gets `min(IL, c·V0)` from the MM, fund covers `(IL − c·V0)⁺`. Aggregate fund inflows (premium share + leverage tax) against outflows (excess IL) across the portfolio; that gives the fund P&L distribution this whole pipeline is converging on.

**Phase 2 cross-check** (Task 2.11, once MSVC + Stylus build).** Pin a handful of `(P_T, Pa, Pb, L, a0, a1)` tuples, call both this module *and* the Stylus contract, and assert agreement to ~1 wei. That is the deliverable that proves the Python quant model and the on-chain math agree.